# Daily Task Documentation - Week 1 Day 1

## Tasks Completed
- **Groq API Integration**: Successfully configured using `llama-3.1-8b-instant`.
- **Decoding Strategy Analysis**:
    - **Temperature**: Used a verbose prompt about a college student to test limits. At `T=0.0`, the model provided a structured, formal narrative. At `T=1.5` and `T=2.0`, the model introduced more specific characters (like 'Reginald P. Bottomsworth') and varied stylistic flourishes.
    - **Top-P**: Observed that low `top-p` (0.1) restricts the vocabulary even at moderate temperatures, resulting in more predictable phrasing.
- **Tokenization Comparison (Tiktoken vs GPT-2)**:
    - **Languages**: Analyzed Arabic and Urdu alongside English and emojis.
    - **Observations**: Modern tokenizers like `tiktoken` (`cl100k_base`) are much more efficient for non-Latin scripts. The Urdu phrase 'آئیے مل کر...' required 37 tokens in Tiktoken compared to 47 in GPT-2.
    - **Finding**: Arabic and Urdu remain 'token-heavy' compared to English, but the transition from older BPE models (GPT-2) to modern ones shows significant optimization for global languages.

In [12]:
# Task 1: Groq API Setup and Decoding Settings
!pip install -q groq tiktoken transformers

import os
from groq import Groq
from google.colab import userdata

# Initialize Groq client
client = Groq(api_key=userdata.get('GROQ_API_KEY'))

def test_decoding(prompt, temp, top_p):
    completion = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=temp,
        top_p=top_p
    )
    return completion.choices[0].message.content

# Run 10 prompts with different settings
settings = [
    (0.0, 1.0), (0.5, 1.0), (1.0, 1.0), (1.5, 1.0), (2.0, 1.0),
    (0.7, 0.1), (0.7, 0.5), (0.7, 0.9), (1.0, 0.5), (0.2, 0.8)
]

print("Running decoding experiments...")
for i, (temp, p) in enumerate(settings):
    try:
        res = test_decoding("Write a verbose joke about a student entering college.", temp, p)
        # Printing full output to see differences in creativity/randomness
        print(f"\n--- Run {i+1} [Temp={temp}, Top-P={p}] ---")
        print(res)
    except Exception as e:
        print(f"Run {i+1} failed: {e}")

Running decoding experiments...

--- Run 1 [Temp=0.0, Top-P=1.0] ---
As the erstwhile high school student, now an as-yet-undefined entity known as a "freshman," stepped onto the hallowed grounds of the esteemed institution of higher learning, a sense of trepidation began to beset their countenance. For it was at this juncture that they were about to embark upon a most arduous and potentially transformative odyssey, one that would necessitate the acquisition of copious amounts of knowledge, the cultivation of novel social relationships, and the development of an altogether more sophisticated and worldly perspective.

And so, as they stood at the threshold of this grand adventure, they were met with a most inauspicious and, indeed, somewhat perturbing query: "Do you have your student ID?" To which they, with a mixture of bewilderment and trepidation, replied: "Ah, yes, I believe I do... somewhere... in my backpack... or perhaps it's in my room... or maybe I left it at home... Oh dear, I 

In [7]:
# Task 2: Tokenization Comparison (Tiktoken vs GPT-2)
import tiktoken
from transformers import AutoTokenizer

# cl100k_base is the tiktoken encoding used by gpt-4 and llama-3
enc_tiktoken = tiktoken.get_encoding("cl100k_base")
enc_gpt2 = AutoTokenizer.from_pretrained("gpt2")

test_strings = [
    "مرحبا بك في عالم الذكاء الاصطناعي", # Arabic
    "آئیے مل کر مصنوعی ذہانت سیکھتے ہیں", # Urdu
    "السلام عليكم", # Arabic Greeting
    "پاکستان زنده باد", # Urdu phrase
    "😊🚀🔥✨", # Emojis
    "def solve(): pass", # Code
    "1234567890", # Numbers
    "A very long English sentence to see if it tokenizes efficiently.",
    "Complex_variable_name_123",
    " ", # Whitespace
    "\n", # Newline
    "gpt-4",
    "tokenization",
    "artificial intelligence",
    "ربوٹ", # Robot in Urdu
    "ذكاء", # Intelligence in Arabic
    "...",
    "!!!",
    "@#$%&*",
    "https://openai.com",
    "email@example.pk",
    "Generative AI",
    "Large Language Models",
    "Training",
    "Day 1",
    "Week 1",
    "BPE",
    "Logits",
    "Temperature",
    "Top-P"
]

print(f"{'String':<35} | {'Tiktoken':<10} | {'GPT-2'}")
print("-" * 60)
for s in test_strings:
    t1 = len(enc_tiktoken.encode(s))
    t2 = len(enc_gpt2.encode(s))
    # Formatting output to handle non-latin script alignment
    display_str = s.replace('\n', '\\n')[:33]
    print(f"{display_str:<35} | {t1:<10} | {t2}")

String                              | Tiktoken   | GPT-2
------------------------------------------------------------
مرحبا بك في عالم الذكاء الاصطناعي   | 23         | 33
آئیے مل کر مصنوعی ذہانت سیکھتے ہی   | 37         | 47
السلام عليكم                        | 9          | 12
پاکستان زنده باد                    | 11         | 18
😊🚀🔥✨                                | 10         | 10
def solve(): pass                   | 4          | 4
1234567890                          | 4          | 4
A very long English sentence to s   | 13         | 13
Complex_variable_name_123           | 5          | 8
                                    | 1          | 1
\n                                  | 1          | 1
gpt-4                               | 4          | 4
tokenization                        | 2          | 2
artificial intelligence             | 3          | 3
ربوٹ                                | 5          | 5
ذكاء                                | 3          | 6
...                         